# Module 6 · Demo — RAG Pipeline

**From 0 to Agentic AI — DataHack Summit 2026**

In Module 2 our bare model **hallucinated** the PTO policy — it had never seen our docs. **RAG**
(Retrieval-Augmented Generation) fixes that: at query time we **retrieve** the relevant text and
**inject** it into the prompt, so the model answers from real, cited context — no retraining.

### The flow
**index** (chunk + embed docs) → **retrieve** (find similar chunks) → **inject** (add to prompt) → **answer**

We build each step on a tiny company-docs corpus, then chain them.

---
## Setup

In [ ]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml.
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" "langgraph>=1.0,<2" \
               "langchain-text-splitters>=0.3" "langchain-chroma>=0.2" "chromadb>=0.5"

In [ ]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Step 1 · The documents

Normally you'd load PDFs / Slack / wiki pages with a **document loader**. To keep the demo
self-contained we hand-write a few `Document`s — the rest of the pipeline is identical.

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(page_content="The billing service is owned by the Payments team. On-call lead: Sam. Escalate outages in #billing-oncall.", metadata={"source": "runbook", "team": "payments"}),
    Document(page_content="Employees get 28 days of paid time off (PTO) per year, plus public holidays. Requests go through the HR portal.", metadata={"source": "hr-policy", "team": "people"}),
    Document(page_content="Production deploys run weekdays at 9pm IST via the release bot. Rollbacks are one command: `deploy rollback <service>`.", metadata={"source": "runbook", "team": "platform"}),
    Document(page_content="Expense reports over $500 need manager approval before submission. Reimbursement takes 5-7 business days.", metadata={"source": "finance-policy", "team": "finance"}),
]
print(len(docs), "documents")

---
## Step 2 · Chunk

Long docs are split into overlapping **chunks**. Chunk size &amp; overlap are the dials that make
or break retrieval: too big = noise, too small = lost meaning.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)
print(len(chunks), "chunks")
print(chunks[0].page_content)

---
## Step 3 · Embed &amp; store (ChromaDB)

Each chunk is turned into an **embedding** (a vector) and stored in **Chroma**. Similar meaning
→ nearby vectors, which is what makes semantic search work.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(chunks, embeddings)
print("indexed", vectorstore._collection.count(), "chunks")

---
## Step 4 · Retrieve

Given a question, embed it and pull the **most similar** chunks. Notice we never told it the
word "PTO" — semantic search finds it anyway.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
hits = retriever.invoke("how much holiday do I get?")
for h in hits:
    print("-", h.page_content[:80], "  <=", h.metadata["source"])

---
## Step 5 · Inject &amp; answer

Put the retrieved chunks into the prompt as **context**, and ask the model to answer *only* from
it (and to cite sources). This is the whole point — the answer is **grounded**.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def answer(question: str) -> str:
    hits = retriever.invoke(question)
    context = "\n\n".join(f"[{h.metadata['source']}] {h.page_content}" for h in hits)
    msgs = [
        SystemMessage("Answer using ONLY the context. Cite the [source]. If unknown, say so."),
        HumanMessage(f"Context:\n{context}\n\nQuestion: {question}"),
    ]
    return llm.invoke(msgs).content

print(answer("How many PTO days do I get, and how do I request them?"))

In [ ]:
print(answer("Who is on-call for billing?"))

In [ ]:
print(answer("What is the CEO's salary?"))   # not in the docs -> should say it doesn't know

---
## Key takeaways
- **RAG changes what the model *knows*** — retrieve relevant text, inject it, answer from it.
- **Chunking + embeddings** decide retrieval quality — most RAG failures are *retrieval* failures.
- Grounding the answer in retrieved context (and citing it) is what kills hallucination.

➡️ **Next (the project):** wrap this retriever as a **`@tool`** so the assistant can *decide* when
to search internal docs — that's **Knowledge Assistant v3**.